The following additional libraries are needed to run this
notebook. Note that running on Colab is experimental, please report a Github
issue if you have any problem.

# 数据预处理
:label:`sec_pandas`

为了能用深度学习来解决现实世界的问题，我们经常从预处理原始数据开始，
而不是从那些准备好的张量格式数据开始。
在Python中常用的数据分析工具中，我们通常使用`pandas`软件包。
像庞大的Python生态系统中的许多其他扩展包一样，`pandas`可以与张量兼容。
本节我们将简要介绍使用`pandas`预处理原始数据，并将原始数据转换为张量格式的步骤。
后面的章节将介绍更多的数据预处理技术。

## 读取数据集

举一个例子，我们首先(**创建一个人工数据集，并存储在CSV（逗号分隔值）文件**)
`../data/house_tiny.csv`中。
以其他格式存储的数据也可以通过类似的方式进行处理。
下面我们将数据集按行写入CSV文件中。


In [4]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名
    f.write('NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

要[**从创建的CSV文件中加载原始数据集**]，我们导入`pandas`包并调用`read_csv`函数。该数据集有四行三列。其中每行描述了房间数量（“NumRooms”）、巷子类型（“Alley”）和房屋价格（“Price”）。


In [11]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
# !pip install pandas
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


## DataFrame的常用函数或者字段

In [6]:
# prompt: data = pd.read_csv(data_file)返回的data是DataFrame对象，我想学习下DataFrame的常用函数或者字段
import pandas as pd
# Assuming 'data' is your DataFrame (from pd.read_csv)
# Example usage with the provided code's context:

# Display the data types of each column
print(data.dtypes)
print("-----------data.dtypes------------")
# Access a specific column
print(data['NumRooms'])
print("-----------data['NumRooms']------------")
# Basic statistics of numerical columns
print(data.describe())
print("-----------data.describe()------------")
# Get the number of rows and columns
print(data.shape)
print("----------data.shape-------------")
# Access the first 5 rows
print(data.head())
print("----------data.head()-------------")
# Access the last 3 rows
print(data.tail(3))
print("----------data.tail(3)-------------")
# Check for missing values
print(data.isnull().sum())
print("-----------data.isnull().sum()------------")
# Fill missing values (e.g., with the mean of the column)
# data['NumRooms'].fillna(data['NumRooms'].mean(), inplace=True)
print("--------Fill missing values---------------")
# Drop rows with missing values
# data.dropna(inplace=True)
print("-----------Drop rows with missing values------------")
# Access specific rows and columns using iloc (integer-based indexing)
print(data.iloc[0:2, 0:2]) # first 2 rows and 2 columns
print("-----------------------")
# Access specific rows and columns using loc (label-based indexing)
print(data.loc[0:2, 'NumRooms':'Alley'])
print("----------Access specific rows and columns using loc-------------")
# Sort the DataFrame by a column
print(data.sort_values(by='Price', ascending=False))
print("-----------Sort------------")
# Group data and calculate aggregate statistics
print(data.groupby('Alley')['Price'].mean())
print("-----------Group data and calculate aggregate statistics------------")
# Apply a function to each element in a column
data['Price'] = data['Price'].apply(lambda x: x * 1.1)
print("---------Apply a function to each element in a column--------------")
# Convert a column to a different data type
# data['NumRooms'] = data['NumRooms'].astype(int)
print("----------Convert a column to a different data type-------------")
# Add a new column
data['NewColumn'] = data['Price'] / 10000
print("------------Add a new column-----------")
data


NumRooms     float64
Alley         object
Price          int64
NewColumn    float64
dtype: object
-----------data.dtypes------------
0    NaN
1    2.0
2    4.0
3    NaN
Name: NumRooms, dtype: float64
-----------data['NumRooms']------------
       NumRooms         Price  NewColumn
count  2.000000       4.00000   4.000000
mean   3.000000  137900.00000  13.790000
std    1.414214   30255.68817   3.025569
min    2.000000  106000.00000  10.600000
25%    2.500000  122125.00000  12.212500
50%    3.000000  133750.00000  13.375000
75%    3.500000  149525.00000  14.952500
max    4.000000  178100.00000  17.810000
-----------data.describe()------------
(4, 4)
----------data.shape-------------
   NumRooms Alley   Price  NewColumn
0       NaN  Pave  127500      12.75
1       2.0   NaN  106000      10.60
2       4.0   NaN  178100      17.81
3       NaN   NaN  140000      14.00
----------data.head()-------------
   NumRooms Alley   Price  NewColumn
1       2.0   NaN  106000      10.60
2       4.0   NaN

,NumRooms,Alley,Price,NewColumn
0,NaN,Pave,140250.0,14.025
1,2.0,NaN,116600.0,11.660
2,4.0,NaN,195910.0,19.591
3,NaN,NaN,154000.0,15.400


## 处理缺失值

注意，“NaN”项代表缺失值。
[**为了处理缺失的数据，典型的方法包括*插值法*和*删除法*，**]
其中插值法用一个替代值弥补缺失值，而删除法则直接忽略缺失值。
在(**这里，我们将考虑插值法**)。

通过位置索引`iloc`，我们将`data`分成`inputs`和`outputs`，
其中前者为`data`的前两列，而后者为`data`的最后一列。
对于`inputs`中缺少的数值，我们用同一列的均值替换“NaN”项。


In [13]:
# print(data)
# print(data["NumRooms"])
# print(type(data["NumRooms"][0]))
# print(data["NumRooms"][0])
# print("--------------------")
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]

print("inputs.id",id(inputs))
# print(inputs)
# # inputs['NumRooms'] = pd.to_numeric(inputs['NumRooms'], errors='coerce')
# # TypeError: can only concatenate str (not "int") to str .Alley 第一行是字符串'Pave'
# inputs['Alley'] = pd.to_numeric(inputs['Alley'], errors='coerce')
# print("after to_numeric. inputs.id",id(inputs))
# print(inputs)
# # data['NumRooms'].fillna(data['NumRooms'].mean(), inplace=True)
# print("inputs.mean() = \n",inputs.mean())
# inputs = inputs.fillna(inputs.mean())
# print("after fillna. inputs.id",id(inputs))
# print(inputs)

inputs.id 137793644398416


[**对于`inputs`中的类别值或离散值，我们将“NaN”视为一个类别。**]
由于“巷子类型”（“Alley”）列只接受两种类型的类别值“Pave”和“NaN”，
`pandas`可以自动将此列转换为两列“Alley_Pave”和“Alley_nan”。
巷子类型为“Pave”的行会将“Alley_Pave”的值设置为1，“Alley_nan”的值设置为0。
缺少巷子类型的行会将“Alley_Pave”和“Alley_nan”分别设置为0和1。


In [15]:
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


In [16]:

inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
print(inputs)
# 旧版本调用
#inputs['NumRooms'].fillna(inputs['NumRooms'].mean(), inplace=True)
# 以下为新版本调用
#inputs.fillna({'NumRooms': 88}, inplace=True)
# inputs['NumRooms'] = inputs['NumRooms'].fillna(inputs['NumRooms'].mean())
# print(inputs)
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms Alley
0       NaN  Pave
1       2.0   NaN
2       4.0   NaN
3       NaN   NaN
   NumRooms  Alley_Pave  Alley_nan
0       NaN        True      False
1       2.0       False       True
2       4.0       False       True
3       NaN       False       True


## 转换为张量格式

[**现在`inputs`和`outputs`中的所有条目都是数值类型，它们可以转换为张量格式。**]
当数据采用张量格式后，可以通过在 :numref:`sec_ndarray`中引入的那些张量函数来进一步操作。


In [17]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y


(tensor([[nan, 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [nan, 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

## 小结

* `pandas`软件包是Python中常用的数据分析工具中，`pandas`可以与张量兼容。
* 用`pandas`处理缺失的数据时，我们可根据情况选择用插值法和删除法。

## 练习

创建包含更多行和列的原始数据集。

1. 删除缺失值最多的列。
2. 将预处理后的数据集转换为张量格式。


[Discussions](https://discuss.d2l.ai/t/1749)
